# Forecasting Hourly Day-Ahead Electricity Prices in the German-Luxembourg Bidding Zone
**DAI Mission — Data & AI in Economics | TU Dortmund**

---

## Team

| Name | Role   |
|------|--------|
| Lennart Oberkönig | Lead   |
| Tim Janis Schmale | Member |

---

**LLM Assistance Disclosure**
  
We used Microsoft Copilot, GitHub Copilot, and ChatGPT to help with selected technical parts of our work, such as setting up the supervised learning pipeline, debugging our own code, and creating clear and visually appealing figures. We also used these tools to smooth out some transitions and improve the overall flow of our written text.
All analyses, interpretations, and conclusions are entirely our own.

## Research Question

*How accurately can hourly day-ahead electricity prices in the German-Luxembourg bidding zone be forecasted using market fundamentals, renewable generation forecasts, load forecasts and calendar effects?*

In [1]:
## Packages

# basic packages
import numpy as np
import pandas as pd
from pathlib import Path
import re
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

# plotting
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns

# sklearn models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import silhouette_score
from sklearn.linear_model import Lasso

# preprocessing and pipelines
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.base import clone

# metrics and model inspection
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# clustering and manifold learning
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# others
import holidays
from itertools import combinations

# Causal Inference
import networkx as nx

np.random.seed(42)
print("All imports successful")

All imports successful


---
## Work Plan

| Section                      | Responsible Member              | Main Tasks                                               |
|------------------------------|---------------------------------|----------------------------------------------------------|
| §1 Research Question & Data  | Lennart Oberkönig               | Data Sourcing, Cleaning, Variable Table                  |
| §2 Causal Inference          | Tim Schmale                     | DAG Design                                               |
| §3 Supervised Learning       | Lennart Oberkönig               | Model Selection, Training, Evaluation                    |
| §4 Unsupervised / Generative | Tim Schmale                     | Method Choice, Implementation, Visualisation             |
| §5 Synthesis & Communication | Lennart Oberkönig & Tim Schmale | Cross-Method Narrative, Conclusion, Notebook Readability |

**Shared tasks:** The whole project was done together in person, but responsibilities have been determined.

---
## Section 1 — Research Question & Data

### Research Question

How accurately can hourly day-ahead electricity prices in the German-Luxembourg bidding zone be forecasted using market fundamentals, renewable generation forecasts, load forecasts and calendar effects?

### Motivation

Electricity price forecasting is a challenging and highly relevant task in modern power markets because short-term electricity prices exhibit complex dynamics and depend on the continuous balance between production and consumption, which is affected by several factors such as demand and weather conditions (Maciejowska, Uniejewski and Weron, 2022, P. 1ff.). In day-ahead electricity markets, market participants submit buy and sell orders for electricity delivery on the following day. These bids and offers are aggregated into demand and supply curves, and the market-clearing price is determined by the intersection of these curves. Thus, the hourly day-ahead price reflects the equilibrium between expected electricity demand and available supply for each delivery hour. The auction for this pricing mechanism closes each day at 12:00 and the prices for the next day are determined (Ghelasi and Ziel, 2024, P. 588f.).

Beyond its methodological relevance, electricity price forecasting also has direct economic value. Accurate day-ahead price forecasts can support market participants in planning bidding strategies, scheduling generation or consumption, managing price risk and identifying economically favorable hours for flexible assets such as storage or demand-side flexibility. In this sense, forecast accuracy is not only a statistical objective but can translate into better market decisions.

The auction-based price formation is closely related to the merit-order effect. Since electricity from renewable energy sources such as wind and solar PV is characterized by negligible marginal costs, increasing renewable feed-in tends to affect the aggregated supply curve and can reduce day-ahead electricity prices (Macedo, Marques and Damette, 2022, P. 885ff.). Therefore, renewable generation is an important explanatory factor for forecasting hourly day-ahead electricity prices. This mechanism is particularly relevant for the German-Luxembourg bidding zone, where electricity prices are closely linked to load and renewable generation. Another driving factor for the price is seasonality, since the price is showing recurring patterns on a weekly, daily and intraday level (Trebbien et al., 2024, P. 35f.). 

From a forecasting perspective, this leads to an important modeling question: whether day-ahead electricity prices should be represented as one continuous hourly time series or as a 24-dimensional daily price vector. In a univariate framework, hourly prices are treated as one high-frequency time series, and forecasts for the 24 hours of the next day are generated sequentially. This means that earlier forecasts can enter the prediction of later hours, which makes the approach sensitive to error accumulation. In contrast, the multivariate framework uses an explicit day-by-hour structure and forecasts all 24 hourly prices of the next day at once. This allows each delivery hour to have its own model structure and to capture hour-specific price patterns (Ziel and Weron, 2018, P. 397ff.). In addition, this framework can be extended by including explanatory variables such as load forecasts, wind and solar generation forecasts and calendar effects. Due to the inclusion of additional explanatory variables we choose a multivariate modeling framework for forecasting the hourly day-ahead prices.

This project combines explanatory and predictive methods to analyze hourly day-ahead electricity prices in the German-Luxembourg bidding zone: a directed acyclic graph is used to structure the assumed relationships between relevant market drivers, K-Means clustering and t-SNE are applied to explore and visualize recurring price regimes, and Decision Tree, Random Forest and Neural Network regression models are evaluated against a naive baseline to assess their forecasting performance.


### Data Sources

| Dataset                          | Source / URL                                                  | Access Method   |
|----------------------------------|---------------------------------------------------------------|-----------------|
| Forecasted Day-Ahead Generation  | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |
| Forecasted Day-Ahead Load        | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |
| Day-Ahead Electricity Price      | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |

### Dataset Information

| Variable                                                                | Type        | Role       | Description                                                                                                                                                                            |
|-------------------------------------------------------------------------|-------------|------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| timestamp                                                               | datetime    | Identifier | Start of timeperiod in Central European (Summer-) Time                                                                                                                                 |
| Total Load FC [MWh]                                                     | float       | feature    | Forecasted total electricity consumption for the following day.                                                                                                                        |
| Wind Offshore Production FC [MWh]                                       | float       | feature    | Forecasted net electricity generation from offshore wind turbines for the following day.                                                                                               |
| Wind Onshore Production FC [MWh]                                        | float       | feature    | Forecasted net electricity generation from onshore wind turbines for the following day.                                                                                                |
| Photovoltaik Production FC [MWh]                                        | float       | feature    | Forecasted net electricity generation from photovoltaic systems for the following day. <br/> The forecast is part of the SMARD category for forecasted wind and photovoltaic generation.     |
| Other Production FC [MWh]                                               | float       | feature    | Forecasted net electricity generation from other systems  for the following day.                                                                                                       |
| Stabilized Day Ahead Price [EUR/MWh]                                    | float       | target     | Hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied.                                                                               |
| German_holiday                                                          | boolean     | feature    | Indicator for whether the calendar date of the timestamp is a public holiday in Germany.                                                                                               |
| Luxembourg_holiday                                                      | boolean     | feature    | Indicator for whether the calendar date of the timestamp is a public holiday in Luxembourg.                                                                                            |
| weekday                                                                 | categorical | feature    | Calendar weekday derived from the timestamp.                                                                                                                                           |
| hour                                                                    | categorical | feature    | Hour of the day derived from the timestamp.                                                                                                                                            |
| month                                                                   | categorical | feature    | Calendar month derived from the timestamp.                                                                                                                                             |
| year                                                                    | numeric     | feature    | Calendar year derived from the timestamp.                                                                                                                                              |
| Lagged Stabilized Day Ahead Price [EUR/MWh] for hour -1,...,-168        | float       | feature    | Hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied for all 24 hours of the last 7 days. <br/>This entry corresponds to 168 lag features in total.     |
| Minimum of Lagged Stabilized Day Ahead Price [EUR/MWh] of day -1,...,-7 | float       | feature    | Minimum of hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied for day -1,...,-7. <br/>This entry corresponds to 7 lag features in total. |
| Maximum of Lagged Stabilized Day Ahead Price [EUR/MWh] of day -1,...,-7 | float       | feature    | Maximum of hourly wholesale electricity price in the day-ahead market with variance stabilization approach applied for day -1,...,-7. <br/>This entry corresponds to 7 lag features in total. |

One row in our dataset represents one hourly observation for the German-Luxembourg bidding zone, including the corresponding day-ahead electricity price, market fundamentals, renewable generation forecasts, load forecast and calendar information for that specific delivery hour.

In [ ]:
## Load data
# notebook working directory
BASE = Path().resolve()
DATA = BASE / "data"

# load the data
load_forecast = pd.read_csv(DATA / "Prognostizierter_Stromverbrauch.csv", delimiter=";")
generation_forecast = pd.read_csv(
    DATA / "Prognostizierte_Erzeugung_Day-Ahead.csv", delimiter=";"
)
day_ahead_price = pd.read_csv(DATA / "Gro_handelspreise.csv", delimiter=";")

# restrict to the desired columns, rename columns & set nans
generation_forecast = generation_forecast[
    [
        "Datum von",
        "Wind Offshore [MWh] Berechnete Auflösungen",
        "Wind Onshore [MWh] Berechnete Auflösungen",
        "Photovoltaik [MWh] Berechnete Auflösungen",
        "Sonstige [MWh] Berechnete Auflösungen",
    ]
].rename(
    columns={
        "Datum von": "timestamp",
        "Wind Offshore [MWh] Berechnete Auflösungen": "Wind Offshore Production FC [MWh]",
        "Wind Onshore [MWh] Berechnete Auflösungen": "Wind Onshore Production FC [MWh]",
        "Photovoltaik [MWh] Berechnete Auflösungen": "Photovoltaic Production FC [MWh]",
        "Sonstige [MWh] Berechnete Auflösungen": "Other Production FC [MWh]",
    }
)

load_forecast = load_forecast[
    ["Datum von", "Netzlast [MWh] Berechnete Auflösungen"]
].rename(
    columns={
        "Datum von": "timestamp",
        "Netzlast [MWh] Berechnete Auflösungen": "Total Load FC [MWh]",
    }
)

day_ahead_price = day_ahead_price[
    ["Datum von", "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen"]
].rename(
    columns={
        "Datum von": "timestamp",
        "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen": "Day Ahead Price [EUR/MWh]",
    }
)

## Prepare data
# Datetime Objects
generation_forecast["timestamp"] = pd.to_datetime(
    generation_forecast["timestamp"], dayfirst=True
)
load_forecast["timestamp"] = pd.to_datetime(load_forecast["timestamp"], dayfirst=True)
day_ahead_price["timestamp"] = pd.to_datetime(
    day_ahead_price["timestamp"], dayfirst=True
)

## Merge Tables (each doubled hour is multiplied 4 times, but will be handled later on anyway with the same results)
df_join = pd.merge(
    pd.merge(load_forecast, generation_forecast, on="timestamp", how="inner"),
    day_ahead_price,
    on="timestamp",
    how="inner",
)

# Numeric columns are created
df_cleaned = df_join.replace("-", np.nan)
cols = df_cleaned.columns[1:]
for col in cols:
    s = (
        df_cleaned[col]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )

    # Convert literal "nan" or empty strings to real NaN
    s = s.replace(["nan", ""], pd.NA)

    # Now safely convert to numeric
    df_cleaned[col] = pd.to_numeric(s, errors="coerce")
df_cleaned

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
0,2018-10-01 02:00:00,42628.00,1750.50,4152.00,0.00,37311.50,51.41
1,2018-10-01 03:00:00,42986.75,1895.25,4436.25,0.00,36318.50,47.38
2,2018-10-01 04:00:00,44675.00,2138.25,4816.25,0.00,37481.50,47.59
3,2018-10-01 05:00:00,48813.25,2368.50,5276.00,0.00,41073.50,51.61
4,2018-10-01 06:00:00,57869.00,2649.25,5625.25,0.00,45581.50,69.13
...,...,...,...,...,...,...,...
67337,2026-06-04 19:00:00,52194.28,4978.72,22366.01,3950.46,25564.73,114.17
67338,2026-06-04 20:00:00,51930.65,5167.91,21943.25,1137.85,25791.74,117.96
67339,2026-06-04 21:00:00,50551.84,5451.33,22037.07,104.24,25573.48,115.37
67340,2026-06-04 22:00:00,48818.86,5682.32,22200.84,0.00,23723.35,111.69


### Data Quality Handling
1. Time shifts (missing and doubled hours)
2. Missing Values
3. Heteroscedastic Variance of Day Ahead Price over the time series.

These quality issues are handled in the following three subsections.

#### Time Shifts

For the time shifts of European time, two central problems appear in our data:
1. Doubled hours, when time is shifted backwards
    - The doubled hour is averaged out of the data (orientation for problem handling by Ziel & Weron, 2018)
2. Missing hours, when time is shifted forwards
    - The missing hour is forward filled (orientation for problem handling by Ziel & Weron, 2018)


Let's take a look at the doubled hours.

In [3]:
# look at duplicated timestamps
dups_ts = df_cleaned["timestamp"].value_counts().loc[lambda x: x > 1].index

df_cleaned.loc[df_cleaned["timestamp"].isin(dups_ts)]

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
648,2018-10-28 02:00:00,NaN,2621.00,10032.50,0.0,34983.50,41.62
649,2018-10-28 02:00:00,NaN,2621.00,10032.50,0.0,34983.50,41.59
650,2018-10-28 02:00:00,NaN,2524.00,10562.00,0.0,34744.00,41.62
651,2018-10-28 02:00:00,NaN,2524.00,10562.00,0.0,34744.00,41.59
652,2018-10-28 02:00:00,NaN,2621.00,10032.50,0.0,34983.50,41.62
...,...,...,...,...,...,...,...
62013,2025-10-26 02:00:00,40283.49,5662.18,33919.70,0.0,16569.90,2.02
62014,2025-10-26 02:00:00,39882.12,5739.19,33457.25,0.0,17222.03,3.19
62015,2025-10-26 02:00:00,39882.12,5739.19,33457.25,0.0,17222.03,2.02
62016,2025-10-26 02:00:00,39882.12,5662.18,33919.70,0.0,16569.90,3.19


Take-Away: each duplicated hour appears 8 times. The data handling of doubled hours will solve this problem.

In [4]:
## Handling of doubled hours via mean
df_time_cleaned = df_cleaned.groupby("timestamp", as_index=False).mean(
    numeric_only=True
)
df_time_cleaned.loc[df_time_cleaned["timestamp"].isin(dups_ts)]

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
648,2018-10-28 02:00:00,NaN,2572.500,10297.250,0.0,34863.750,41.605
9383,2019-10-27 02:00:00,38813.750,5672.375,27504.125,0.0,19140.000,-19.970
18118,2020-10-25 02:00:00,38046.750,5299.000,24125.750,0.0,19260.250,0.120
27021,2021-10-31 02:00:00,42149.125,2995.625,16252.500,0.0,20583.375,66.760
35756,2022-10-30 02:00:00,42986.750,1517.750,7816.625,0.0,25391.625,100.060
44491,2023-10-29 02:00:00,37033.625,6074.750,23126.500,0.0,NaN,0.015
53226,2024-10-27 02:00:00,37638.875,2071.375,9575.750,0.0,NaN,81.330
61961,2025-10-26 02:00:00,40082.805,5700.685,33688.475,0.0,16895.965,2.605


Let's take a look at the missing hours.

In [5]:
# keep original timestamps to detect newly inserted rows later
original_timestamps = df_time_cleaned["timestamp"].copy()

# create a complete hourly time index
full_range = pd.date_range(
    start=df_time_cleaned["timestamp"].min(),
    end=df_time_cleaned["timestamp"].max(),
    freq="h",
)

# reindex to full hourly range (introduces NaNs for missing timestamps)
df_time_cleaned = df_time_cleaned.set_index("timestamp").reindex(full_range)
df_time_cleaned.index.name = "timestamp"

# identify rows that did not exist before (DST gaps)
new_rows = ~df_time_cleaned.index.isin(original_timestamps)

# look at missing timestamps
df_time_cleaned[~df_time_cleaned.index.isin(original_timestamps)]

,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
timestamp,,,,,,
2019-03-31 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2020-03-29 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2021-03-28 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-27 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2023-03-26 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-31 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-30 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-29 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# forward-fill all values
df_ffill = df_time_cleaned.ffill()

# fill only the newly inserted timestamps with forward-filled values
df_time_cleaned.loc[new_rows, :] = df_ffill.loc[new_rows, :]

# restore timestamp as a column
df_time_cleaned = df_time_cleaned.reset_index()

# look at missing timestamps
df_time_cleaned[~df_time_cleaned.index.isin(original_timestamps)]

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh]
0,2018-10-01 02:00:00,42628.00,1750.50,4152.00,0.00,37311.50,51.41
1,2018-10-01 03:00:00,42986.75,1895.25,4436.25,0.00,36318.50,47.38
2,2018-10-01 04:00:00,44675.00,2138.25,4816.25,0.00,37481.50,47.59
3,2018-10-01 05:00:00,48813.25,2368.50,5276.00,0.00,41073.50,51.61
4,2018-10-01 06:00:00,57869.00,2649.25,5625.25,0.00,45581.50,69.13
...,...,...,...,...,...,...,...
67289,2026-06-04 19:00:00,52194.28,4978.72,22366.01,3950.46,25564.73,114.17
67290,2026-06-04 20:00:00,51930.65,5167.91,21943.25,1137.85,25791.74,117.96
67291,2026-06-04 21:00:00,50551.84,5451.33,22037.07,104.24,25573.48,115.37
67292,2026-06-04 22:00:00,48818.86,5682.32,22200.84,0.00,23723.35,111.69


Take-Away: Missing and double hours due to time shifting are eliminated from the dataset in such a way that all timestamp are completely unique.

#### Missing Values

For the possible imputation of values, we introduce some seasonality features into the data set through feature engineering.

In [7]:
## first feature engineering
# Holidays
de = holidays.DE()
lu = holidays.LU()
df_time_cleaned["date only"] = df_time_cleaned["timestamp"].dt.date
df_time_cleaned["German_holiday"] = df_time_cleaned["date only"].apply(
    lambda x: x in de
)
df_time_cleaned["Luxembourg_holiday"] = df_time_cleaned["date only"].apply(
    lambda x: x in lu
)
df_time_cleaned.drop(columns=["date only"], inplace=True)

# Timestamp patterns
df_time_cleaned["weekday"] = df_time_cleaned["timestamp"].dt.weekday.astype("category")
df_time_cleaned["hour"] = df_time_cleaned["timestamp"].dt.hour.astype("category")
df_time_cleaned["month"] = df_time_cleaned["timestamp"].dt.month.astype("category")
df_time_cleaned["year"] = df_time_cleaned["timestamp"].dt.year.astype("int")

In [8]:
# check for empty values
df_time_cleaned.isna().sum()

timestamp                               0
Total Load FC [MWh]                  1033
Wind Offshore Production FC [MWh]       0
Wind Onshore Production FC [MWh]        3
Photovoltaik Production FC [MWh]        3
Other Production FC [MWh]            1688
Day Ahead Price [EUR/MWh]               0
German_holiday                          0
Luxembourg_holiday                      0
weekday                                 0
hour                                    0
month                                   0
year                                    0
dtype: int64

To address the missing values in all 4 features, we perform data imputation using the kNN method. This way, the empty values are set to a more realistic value compared to applications of forward fill or other methods, especially when multiple values are missing continuously.

In [9]:
# define the columns which have empty values
impute_targets = df_time_cleaned.columns[df_time_cleaned.isna().any()]

# define the columns used for prediction
predictor_cols = df_time_cleaned.columns[
    ~df_time_cleaned.columns.isin(["timestamp", "Day Ahead Price [EUR/MWh]"])
]

# working copy
df = df_time_cleaned.copy()
# cache for fitted models
model_cache = {}

# imputation by kNN
for target in impute_targets:

    # do not include target column in predictors
    candidate_predictors = [col for col in predictor_cols if col != target]

    # get all rows where the target is missing
    missing_indices = df_time_cleaned.index[df_time_cleaned[target].isna()]

    if len(missing_indices) == 0:
        print(f"{target}: no missing values")
        continue

    # iterate over empty rows
    for idx in missing_indices:

        # get all available predictors (non-NaN) for this row
        available_predictors = [
            col
            for col in candidate_predictors
            if pd.notna(df_time_cleaned.loc[idx, col])
        ]

        # at least one predictor must be available
        if len(available_predictors) == 0:
            continue

        predictors = tuple(available_predictors)

        # training rows: target present + predictors present
        train_mask = df_time_cleaned[target].notna() & df_time_cleaned[
            list(predictors)
        ].notna().all(axis=1)

        X_train = df_time_cleaned.loc[train_mask, list(predictors)]
        y_train = df_time_cleaned.loc[train_mask, target]
        # skip when not enough train data is available
        if len(X_train) < 2:
            continue

        # build model if not cached
        cache_key = (target, predictors)

        if cache_key not in model_cache:
            k = min(5, len(X_train))
            knn = Pipeline(
                [
                    ("scaler", StandardScaler()),
                    ("knn", KNeighborsRegressor(n_neighbors=k, weights="distance")),
                ]
            )
            knn.fit(X_train, y_train)
            model_cache[cache_key] = knn

        # predict with kNN
        X_pred = df_time_cleaned.loc[[idx], list(predictors)]
        prediction = model_cache[cache_key].predict(X_pred)[0]

        df.loc[idx, target] = prediction

print(df[impute_targets].isna().sum())

Total Load FC [MWh]                 0
Wind Onshore Production FC [MWh]    0
Photovoltaik Production FC [MWh]    0
Other Production FC [MWh]           0
dtype: int64


Take Away: All missing values are eliminated.

#### Variance Stabilization

As energy time-series have the potential to have heteroscedastic variances over time (Ziel and Weron, 2018), we take a look at the time-series to analyze the need for stabilization.

In [10]:
# select price column
col = "Day Ahead Price [EUR/MWh]"

# build line plot before price stabilization
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(df["timestamp"], df[col], color="royalblue", linewidth=2)

ax.set_title(f"{col} prior variance stabilization", fontsize=16, pad=12)
ax.set_xlabel("time")
ax.set_ylabel(col)

plt.tight_layout()
fname = f"plots/Day_Ahead_Price_EUR_MWh_prior_stabilization.png"

plt.savefig(fname, dpi=100, bbox_inches="tight")
plt.close()
print(f"{col} saved \u2192 {fname}")

Day Ahead Price [EUR/MWh] saved → plots/Day_Ahead_Price_EUR_MWh_prior_stabilization.png


Take-Away:
- Heteroscedastic variance over time series are apparent
- This needs to be handled via variance stabilization

Let's stabilize the prices based on the method given by Ziel and Weron (2018).

In [11]:
# select the price column
col = "Day Ahead Price [EUR/MWh]"

# sort data chronologically
df_sorted = df.sort_values("timestamp").reset_index(drop=True)

# set parameters for the first 730-day window
start_ts = df_sorted["timestamp"].min()
end_ts = start_ts + pd.Timedelta(days=730)

first_window = df_sorted[df_sorted["timestamp"] < end_ts]

# calculate median and MAD for the first window
a = first_window[col].median()
b = 1.4826 * np.median(np.abs(first_window[col] - a))

# create new column for stabilized price
df_sorted["Stabilized Day Ahead Price [EUR/MWh]"] = np.nan

# dictionary for later inverse transformation
transform_params = {}

# transform the first window
mask_first = df_sorted["timestamp"] < end_ts
df_sorted.loc[mask_first, "Stabilized Day Ahead Price [EUR/MWh]"] = np.arcsinh(
    (df_sorted.loc[mask_first, col] - a) / b
)

# save parameters for all timestamps of the first window (for later inverse transformation)
first_days = df_sorted.loc[mask_first, "timestamp"].dt.floor("D").unique()
for d in first_days:
    transform_params[pd.Timestamp(d)] = (a, b)

# iterate over all subsequent days and apply rolling window transformation
unique_days = df_sorted["timestamp"].dt.floor("D").unique()
window_days = 730

for i in range(window_days, len(unique_days)):

    # train window
    train_start = unique_days[i - window_days]
    train_end = unique_days[i]

    train = df_sorted[
        (df_sorted["timestamp"] >= train_start) & (df_sorted["timestamp"] < train_end)
    ]

    # parameters
    a_i = train[col].median()
    b_i = 1.4826 * np.median(np.abs(train[col] - a_i))
    if b_i == 0:
        b_i = 1e-6

    # test day
    test_day = unique_days[i]
    mask_test = df_sorted["timestamp"].dt.floor("D") == test_day

    # transform test day
    df_sorted.loc[mask_test, "Stabilized Day Ahead Price [EUR/MWh]"] = np.arcsinh(
        (df_sorted.loc[mask_test, col] - a_i) / b_i
    )

    # save parameters for the test day (for later inverse transformation)
    transform_params[pd.Timestamp(test_day)] = (a_i, b_i)

df = df_sorted.copy()

In [12]:
# select stabilized price column
col = df.columns[-1]

# build line plot after price stabilization
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(df["timestamp"], df[col], color="royalblue", linewidth=2)

ax.set_title(
    f"Day Ahead Price [EUR/MWh] after variance stabilization", fontsize=16, pad=12
)
ax.set_xlabel("time")
ax.set_ylabel(col)

plt.tight_layout()
fname = f"plots/Day_Ahead_Price_EUR_MWh_after_stabilization.png"

plt.savefig(fname, dpi=100, bbox_inches="tight")
plt.close()
print(f"Day Ahead Price [EUR/MWh] after variance stabilization saved \u2192 {fname}")

Day Ahead Price [EUR/MWh] after variance stabilization saved → plots/Day_Ahead_Price_EUR_MWh_after_stabilization.png


Take-Away:
After the iterative application of the variance stabilization method based on MAD and median of the 730 day training phase as well as the asinh function, the time series chart of the DE-LU Day-Ahead Energy price looks way less spiky. Based on this, we will look at the distributions prior and after the variance stabilization.

In [13]:
# create histogram of day ahead price
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(
    df["Day Ahead Price [EUR/MWh]"],
    bins=100,
    color="steelblue",
    edgecolor="white",
    alpha=0.9,
)

ax.set_title("Distribution of Day-Ahead Price [€/MWh]", fontsize=16, pad=12)
ax.set_xlabel("Day-Ahead Price [€/MWh]")
ax.set_ylabel("Count")

ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    "plots/day_ahead_price_distribution_prior_stabilization.png",
    dpi=100,
    bbox_inches="tight",
)
plt.close()
print(
    "Day-Ahead Price distribution prior stabilization saved \u2192 plots/day_ahead_price_distribution_prior_stabilization.png"
)

Day-Ahead Price distribution prior stabilization saved → plots/day_ahead_price_distribution_prior_stabilization.png


In [14]:
# create histogram of stabilized day ahead price
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(
    df["Stabilized Day Ahead Price [EUR/MWh]"],
    bins=100,
    color="steelblue",
    edgecolor="white",
    alpha=0.9,
)

ax.set_title("Distribution of Stabilized Day-Ahead Price [€/MWh]", fontsize=16, pad=12)
ax.set_xlabel("Stabilized Day-Ahead Price [€/MWh]")
ax.set_ylabel("Count")

ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    "plots/stabilized_day_ahead_price_distribution.png", dpi=100, bbox_inches="tight"
)
plt.close()
print(
    "Stabilized Day-Ahead Price distribution saved \u2192 plots/stabilized_day_ahead_price_distribution.png"
)

Stabilized Day-Ahead Price distribution saved → plots/stabilized_day_ahead_price_distribution.png


Take-Away:
- Distribution of prices is rather left skewed on the positive side of 0.
- positive and negative price outliers exist with positive outliers being more frequent than negative outliers.

### Additional Feature Engineering

For the supervised learning part, some lag features are introduced which can be used in all different models, but especially in the Autoregressive with Exogenous Input Model.

In [15]:
# set column names
timestamp_col = "timestamp"
target_col = "Stabilized Day Ahead Price [EUR/MWh]"

# prepare timestamp and sort
df[timestamp_col] = pd.to_datetime(df[timestamp_col])
df = df.sort_values(timestamp_col).reset_index(drop=True)
df["date"] = df[timestamp_col].dt.floor("D")

# pivot prices by date and hour
price_by_day_hour = df.pivot_table(
    index="date", columns="hour", values=target_col, aggfunc="first"
).sort_index()

# create lagged day-hour price features for last 7 days
lag_blocks = []

for d in range(1, 8):
    lag_block = price_by_day_hour.shift(d)
    lag_block.columns = [f"price_d-{d}_h{int(col)}" for col in lag_block.columns]
    lag_blocks.append(lag_block)

day_hour_lags = pd.concat(lag_blocks, axis=1)

# daily min/max from previous days
daily_stats = price_by_day_hour.agg(["min", "max"], axis=1)
daily_stats.columns = ["daily_min", "daily_max"]

daily_lag_blocks = []

for d in range(1, 8):
    daily_lag = daily_stats.shift(d).copy()
    daily_lag.columns = [f"{col}_d-{d}" for col in daily_lag.columns]
    daily_lag_blocks.append(daily_lag)

daily_lags = pd.concat(daily_lag_blocks, axis=1)

# combine all lag features
lag_features = pd.concat([day_hour_lags, daily_lags], axis=1)

# merge back to original df
df_final = df.merge(lag_features, left_on="date", right_index=True, how="left")

# cleanup
df_final = df_final.drop(columns=["date"])
df_final = df_final.dropna().reset_index(drop=True)

df_final

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh],German_holiday,Luxembourg_holiday,weekday,...,daily_min_d-3,daily_max_d-3,daily_min_d-4,daily_max_d-4,daily_min_d-5,daily_max_d-5,daily_min_d-6,daily_max_d-6,daily_min_d-7,daily_max_d-7
0,2018-10-09 00:00:00,47590.250000,3053.00,4436.75,0.00,44412.25,56.77,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
1,2018-10-09 01:00:00,46000.750000,3110.00,4333.50,0.00,43724.50,57.43,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
2,2018-10-09 02:00:00,45662.512065,3180.00,4231.00,0.00,43023.00,55.11,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
3,2018-10-09 03:00:00,45362.156474,3249.25,4217.00,0.00,42405.75,52.35,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
4,2018-10-09 04:00:00,47994.961623,3308.25,4233.75,0.00,43680.00,54.86,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.99713,-1.773498,1.631451,-1.624609,1.602742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67099,2026-06-04 19:00:00,52194.280000,4978.72,22366.01,3950.46,25564.73,114.17,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263
67100,2026-06-04 20:00:00,51930.650000,5167.91,21943.25,1137.85,25791.74,117.96,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263
67101,2026-06-04 21:00:00,50551.840000,5451.33,22037.07,104.24,25573.48,115.37,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263
67102,2026-06-04 22:00:00,48818.860000,5682.32,22200.84,0.00,23723.35,111.69,False,False,3,...,-0.187284,2.713492,-1.724049,1.346221,-1.740193,1.55129,-1.727570,2.153582,-1.728640,2.861263


### Data Investigation

Prior to the data modeling part, the data is investigated using correlation matrix and time-series plots.

#### Correlation Matrix

In [16]:
# build correlation matrix
cols = df.columns.drop("timestamp")[:13]

corr_matrix = df[cols].corr()

# build the plot
plt.figure(figsize=(16, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"},
)
plt.title("Correlation Matrix", fontsize=18, pad=20)
plt.tight_layout()
plt.savefig("plots/corr_matrix.png", dpi=100, bbox_inches="tight")
plt.close()
print("Correlation matrix saved \u2192 plots/corr_matrix.png")

Correlation matrix saved → plots/corr_matrix.png


Take-Away:
- The two wind production forecasts have a strong positive correlation, which is expected given that both are driven by similar meteorological conditions.
- Load and total production are positively correlated, reflecting typical market dynamics: higher demand requires higher generation levels.
- The positive correlation between day‑ahead prices and conventional (non‑renewable) production suggests that prices tend to rise when more traditional generation is required.
- The slightly positive correlation between the year variable and renewable production indicates a structural increase in renewable generation capacity over time.
- This trend is underlined by the negative correlation between the year variable and conventional production.
- The positive correlation between day‑ahead prices and the year variable points to a long‑term upward trend in electricity prices, potentially driven by inflation, geopolitical events, and broader market disruptions such as wars or pandemics.


#### Time-Series Plots

In [17]:
plt.style.use("seaborn-v0_8-whitegrid")

# restrict to relevant columns
cols = df.columns.drop("timestamp")[:4]

# build line plots iteratively
for col in cols:
    fig, ax = plt.subplots(figsize=(16, 4))

    ax.plot(df["timestamp"], df[col], color="royalblue", linewidth=2)

    ax.set_title(f"{col} over time", fontsize=16, pad=12)
    ax.set_xlabel("time")
    ax.set_ylabel(col)

    plt.tight_layout()
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", col).strip("_")
    fname = f"plots/{safe_name}_over_time.png"

    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.close()
    print(f"{col} saved \u2192 {fname}")

Total Load FC [MWh] saved → plots/Total_Load_FC_MWh_over_time.png
Wind Offshore Production FC [MWh] saved → plots/Wind_Offshore_Production_FC_MWh_over_time.png
Wind Onshore Production FC [MWh] saved → plots/Wind_Onshore_Production_FC_MWh_over_time.png
Photovoltaik Production FC [MWh] saved → plots/Photovoltaik_Production_FC_MWh_over_time.png


Take-Away:
- The time-series plots indicate expected seasonality patterns for load and production figures.
- The Price time series as underlines the slightly increasing price mean over time.
- Also, the price variance gets larger as the years go on, this is why the the variance stabilization is applied.
- The variance stabilization is shown in the second price time series.

### Negative Price Distribution

In [18]:
# marker if price is negative or positive
df["negative_price_flag"] = (df["Day Ahead Price [EUR/MWh]"] < 0).astype(int)

# build distribution by hour of day
neg_dist = df.groupby(["hour", "negative_price_flag"], observed=False).size().unstack()
neg_dist = neg_dist.rename(columns={0: "Positive prices", 1: "Negative prices"})


# convert to long format for plotly
neg_dist_plot = neg_dist.reset_index().melt(
    id_vars="hour",
    value_vars=["Positive prices", "Negative prices"],
    var_name="type",
    value_name="count",
)

hours = neg_dist.index
pos = neg_dist["Positive prices"]
neg = neg_dist["Negative prices"]

# stacked bar plot for positive and negative price distribution by hour
plt.figure(figsize=(14, 5))
plt.bar(hours, pos, label="Positive prices", color="steelblue")
plt.bar(hours, neg, bottom=pos, label="Negative prices", color="firebrick")

plt.title("Positive and Negative Prices by Hour", fontsize=16, pad=12)
plt.xlabel("Hour of Day")
plt.ylabel("Number of Observations")
plt.xticks(range(24))
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend()

plt.tight_layout()
plt.savefig("plots/negative_price_distribution.png", dpi=100, bbox_inches="tight")
plt.close()

print("Negative Price distribution saved \u2192 plots/negative_price_distribution.png")

# drop column again
df.drop(columns=["negative_price_flag"], inplace=True)

Negative Price distribution saved → plots/negative_price_distribution.png


Take-Away:
- Negative prices seem to occur mostly around midday which suits the thesis that renewable energies, especially Photovoltaic are causing this phenomenon

---
## Section 2 — Causal Inference Block

In this section, we investigate the causal structures of our outcome variable, the Day-Ahead Electricity Price, using a directed acyclic graph (DAG).
The graph visualizes both the relationships in our dataset and the broader causal mechanisms known from the electricity markets.
Here, solid arrows represent relationships originated from observed variables, while dashed arrows indicate from unobserved or latent variables of our causal model.
In summary, the DAG is used as a conceptional framework that describes and visualizes the causal roles of all variables and reveals the limits of causal identifiability in our setting. This directly determines the limitations and implications on modeling choice and interpretation for the supervised learning part.


In [19]:
# build graph
G = nx.DiGraph()

# solid edges = observed
solid_edges = [
    ("seasonality", "load_forecast"),
    ("holidays", "load_forecast"),
    ("forecasted_supply_renewables", "supply_forecast"),
    ("forecasted_supply_traditional_powerplants", "supply_forecast"),
    ("load_forecast", "day_ahead_price"),
    ("supply_forecast", "day_ahead_price"),
]

# Dashed edges = unobserved
dashed_edges = [
    ("weather_forecast", "load_forecast"),
    ("weather_forecast", "forecasted_supply_renewables"),
    ("prices_traditional_energy_sources", "forecasted_supply_traditional_powerplants"),
    ("industrial_activities", "load_forecast"),
    ("downtimes_powerplants", "forecasted_supply_traditional_powerplants"),
    ("downtimes_powerplants", "forecasted_supply_renewables"),
    ("cross_border_flows", "supply_forecast"),
]

G.add_edges_from(solid_edges + dashed_edges)

# Node positions
pos = {
    "holidays": (2.4, -1.0),
    "industrial_activities": (0.7, 0.0),
    "seasonality": (0.7, 1.5),
    "weather_forecast": (2.4, 1.5),
    "load_forecast": (2.4, 0.0),
    "day_ahead_price": (4.4, 0.0),
    "cross_border_flows": (6.4, -1.0),
    "prices_traditional_energy_sources": (7.8, -1.0),
    "supply_forecast": (6.4, 0.0),
    "forecasted_supply_renewables": (7.8, 1.5),
    "forecasted_supply_traditional_powerplants": (7.8, 0.0),
    "downtimes_powerplants": (9.3, 0),
}

# Node labels
labels = {
    "seasonality": "Seasonality",
    "weather_forecast": "Weather Forecast",
    "holidays": "Holidays",
    "industrial_activities": "Industrial\nActivities",
    "load_forecast": "Load\nForecast",
    "supply_forecast": "Supply\nForecast",
    "forecasted_supply_renewables": "Forecasted Supply\nRenewables",
    "forecasted_supply_traditional_powerplants": "Forecasted Supply\nTraditional\nPowerplants",
    "downtimes_powerplants": "Downtimes of\nPowerplants",
    "prices_traditional_energy_sources": "Prices of Traditional\nEnergy Sources",
    "cross_border_flows": "Cross-Border\nFlows",
    "day_ahead_price": "Day-Ahead\nPrice",
}

# node colors
node_colors = [
    (
        "#FDEBD0"
        if n
        in (
            "seasonality",
            "holidays",
            "industrial_activities",
            "weather_forecast",
            "prices_traditional_energy_sources",
            "downtimes_powerplants",
            "cross_border_flows",
        )
        else (
            "#F9E79F"
            if n
            in (
                "load_forecast",
                "supply_forecast",
                "forecasted_supply_renewables",
                "forecasted_supply_traditional_powerplants",
            )
            else "#A9DFBF"
        )
    )
    for n in G.nodes()
]


# plot
fig, ax = plt.subplots(figsize=(14, 6))

# draw nodes
nx.draw_networkx_nodes(
    G,
    pos=pos,
    ax=ax,
    node_color=node_colors,
    node_size=5000,
    edgecolors="dimgray",
    linewidths=1,
    margins=0.1,
)

# draw labels
nx.draw_networkx_labels(G, pos=pos, labels=labels, ax=ax, font_size=7.5)

# draw solid edges
nx.draw_networkx_edges(
    G,
    pos=pos,
    edgelist=solid_edges,
    ax=ax,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=22,
    edge_color="dimgray",
    width=1.6,
    style="solid",
    min_source_margin=25,
    min_target_margin=30,
)

# draw dashed edges
nx.draw_networkx_edges(
    G,
    pos=pos,
    edgelist=dashed_edges,
    ax=ax,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=22,
    edge_color="dimgray",
    width=1.6,
    style="dashed",
    min_source_margin=25,
    min_target_margin=30,
)

# finalize
ax.set_title("Causal DAG: Drivers of Day-Ahead Electricity Prices", fontsize=12)
ax.axis("off")

plt.tight_layout()
plt.savefig("plots/dag.png", dpi=150, bbox_inches="tight")
plt.close()

print("DAG saved \u2192 plots/dag.png'")

DAG saved → plots/dag.png'


### Interpretation

Looking at our causal graph for the Day-Ahead Electricity Price [in €/MWh] of the German-Luxembourg Bidding Zone, we can observe the following main components:
1. Observed vs. Unobserved Variables
    - Observed: Holidays, Seasonality, Load Forecast, Supply Forecast, Renewables Supply Forecast, Traditional Powerplant Forecast
    - Unobserved: Industrial Activities, Weather Forecast, Cross-Border Flows, Downtime of Powerplants, Prices of Traditional Energy Sources
2. Causal Role
    - Confounders
        - Weather Forecast
    - Colliders
        - Supply Forecast, Load Forecast, Day-Ahead Price
    - Mediators
        - Renewables Supply Forecast, Forecasted Supply of Traditional Powerplants
    - Instruments
        - None Available
3. Backdoor Paths and Identifiability
    - The causal graph gives us two backdoor paths
        1. Price <- Load <- Weather -> Renewables -> Supply -> Price
        2. Price <- Load <- Weather -> Supply -> Price
    - Both paths are caused by the weather forecast which acts as a common cause of both load and supply.
    - The consequence of this is that the observed association between load/supply and the outcome variable price is not equivalent to the causal effect.
    - In general, such backdoor paths need to be closed which can be handled through observing or conditioning on the confounding variable of the weather forecast.
    - However, as the weather forecast is not observed in our data scenario, the backdoor path remains open as it cannot be conditioned on.
    - Further, a backdoor-adjustment is not possible in our DAG, as the minimal set of features that would block both backdoor paths consists solely of the unobserved weather forecast itself.
    - In summary, the causal effects of the load and supply forecasts on the day-ahead price cannot be identified as open backdoor paths remain and cannot be closed in the given data scenario. This results in the fact that our estimated relationships need to be viewed as associative instead of causal and our supervised learning is rather prediction-oriented instead of causality-oriented.
4. Economic Interpretation
    - The day-ahead price is built based on the typical principle of demand vs. supply in this case by load forecast vs. supply forecast. When a high load or a low supply are forecasted, we expect the price to rise, while vice versa is expected to decrease the day-ahead price.
    - Looking at the load forecast in more detail reveals that it is influenced by several factors. One of these factors is seasonality with patterns like hour of the day, weekday, or month of the year, where for example, night hours or weekends are expected to have a reduced load. Further factors are holidays and industrial activities which influence the load as well, as holidays are expected to have reduced loads and hours with high industrial activities increase the energy demand.
    - For the forecasted supply, we determined three key influencing factors. In detail, these are the cross border flows of imported and exported energy, the renewables forecasted supply, and the forecasted supply of traditional powerplants. For all these influencing factors, an increase of their supply increases the total energy supply forecast. In more detail, the downtime of powerplants influences the forecasts of renewable and traditional energy powerplants, while the traditional supply is also influenced by the prices of the traditional energy sources.
    - Last but not least, both the load and the supply forecast are influenced by the weather forecast. For the load forecast, very cold or hot weather leads to an increase of energy demand for heating and cooling systems, while for the supply forecast, the forecast of renewables is increasing with stronger wind conditions or increased sun hours.

5. Limitations and Implications for following parts
    - Unobserved influencing factors
        - Components such as industrial activities, cross‑border flows, and downtimes of powerplants are not observed and therefore stay latent in this study.
    - Unclosed backdoor paths
        - The confounder of weather forecast is unobserved which creates two unclosed backdoor paths.
        - This leads to not identifiable effects of the supply and load forecast on the day-ahead price, which can therefore not be treated as causal but rather associative.
    - Prediction-oriented study
        - Given the unclosed backdoor paths, this study sets their focus rather on the predictive performance of the supervised learning part instead of the causal explanations.
    - Implications on model choice and interpretation
        - Given the prediction-oriented study, flexible ML-models instead of causal-driven models are preferred in the supervised learning part. This leads to our model choice of decision trees, random forest, and neural networks.
        - Further, the supervised learning results should not be interpreted as causal relationships, but rather as statistical patterns which are useful for day-ahead price prediction.
    

---

## Section 3 — Supervised Learning Block

---

In this section, we aim to forecast hourly day-ahead electricity prices in the German-Luxembourg Bidding Zone using supervised learning techniques. Based on the causal inference section, the included variables are interpreted as predictive factors rather than fully causal drivers, since relevant confounders such as weather conditions, fuel prices, cross-border flows, and power plant outages are not completely observed. Therefore, this block focuses on forecasting performance rather than causal identification. For this, the following pipeline is implemented:

1. Preparation of Hourly Forecasting Setup

   * Splitting the dataset into 24 separate subsets, one for each hour of the day, to enable the model to capture hour-specific price dynamics and intraday market patterns.

2. Model Inputs

   * Using market fundamentals such as load forecasts, wind offshore forecasts, wind onshore forecasts, photovoltaic forecasts, and other production forecasts.
   * Using addtionaly calendar variables including weekday, month, year, and holiday indicators.
   * Including lagged price features to capture the autoregressive structure of electricity prices.

3. Rolling Forecasting Approach

   * Applying a rolling forecasting framework with a two-year training window.
   * Predicting the hourly day-ahead price for the following test day.
   * Repeating this procedure across the full test period to evaluate model performance under realistic time-series conditions.

4. Baseline Models and Machine Learning Models

   * Including a Naive baseline, which uses the last available observed price as a simple persistence-based benchmark.
   * Including an ARX model as an additional statistical baseline that combines autoregressive price information with exogenous input variables.
   * Estimating machine learning models including a Decision Tree, Random Forest, and Neural Network.
   * This setup allows us to evaluate whether more flexible machine learning models provide additional forecasting value compared to both a simple naive benchmark and a more structured autoregressive baseline.

5. Model Estimation and Hyperparameter Optimization

   * Performing hyperparameter optimization on the first rolling interval using time-series cross-validation.
   * Reusing the optimized hyperparameters for the remaining rolling forecasts to keep the procedure computationally feasible.

6. Evaluation and Interpretation of Forecasting Results

   * Evaluating the models using MAE and RMSE
   * Comparing the machine learning models against the Naive baseline and the ARX baseline to assess whether they provide additional predictive value.
   * Interpreting the results with respect to forecasting accuracy, economic relevance, and remaining limitations.


### Modeling

In [20]:
# create one df per hour of day
hourly_dfs = {}

for hour in range(24):
    hourly_dfs[hour] = df_final[df_final["hour"] == hour].copy()

# look at one example
hourly_dfs[0].head()

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh],German_holiday,Luxembourg_holiday,weekday,...,daily_min_d-3,daily_max_d-3,daily_min_d-4,daily_max_d-4,daily_min_d-5,daily_max_d-5,daily_min_d-6,daily_max_d-6,daily_min_d-7,daily_max_d-7
0,2018-10-09,47590.250000,3053.00,4436.75,0.0,44412.25,56.77,False,False,1,...,0.668194,1.883336,0.678337,1.897518,0.618903,1.997130,-1.773498,1.631451,-1.624609,1.602742
24,2018-10-10,47056.258035,1653.50,3492.50,0.0,45229.00,53.70,False,False,2,...,0.430020,1.855446,0.668194,1.883336,0.678337,1.897518,0.618903,1.997130,-1.773498,1.631451
48,2018-10-11,49332.677681,4619.25,18584.50,0.0,32970.25,35.01,False,False,3,...,0.716108,2.227772,0.430020,1.855446,0.668194,1.883336,0.678337,1.897518,0.618903,1.997130
72,2018-10-12,50265.750000,2824.25,12392.75,0.0,37070.00,45.85,False,False,4,...,0.969730,2.343601,0.716108,2.227772,0.430020,1.855446,0.668194,1.883336,0.678337,1.897518
96,2018-10-13,47562.500000,4277.25,13403.75,0.0,33339.00,47.80,False,False,5,...,-0.213461,2.088352,0.969730,2.343601,0.716108,2.227772,0.430020,1.855446,0.668194,1.883336


In [21]:
"""
# settings
SCORING = "neg_mean_absolute_error"
HOUR_N_JOBS = 12
GRID_N_JOBS = 2
RF_N_JOBS = 4
PERM_N_JOBS = 4
N_SPLITS = 3

GRID_SEARCH_SPACES = {

    "Decision Tree": {
        "estimator": DecisionTreeRegressor(
            random_state=42
        ),
        "param_grid": {
            "max_depth": [3, 5, 10, None],
            "min_samples_leaf": [1, 5, 10, 20],
            "min_samples_split": [2, 10, 20]
        }
    },

    "ARX": {
        "estimator": Pipeline(steps=[
            ("scaler", StandardScaler()),
            ("model", Lasso(
                random_state=42,
                max_iter=10000
            ))
        ]),
        "param_grid": {
            "model__alpha": [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
        }
    },

    "Random Forest": {
        "estimator": RandomForestRegressor(
            random_state=42,
            n_jobs=RF_N_JOBS
        ),
        "param_grid": {
            "n_estimators": [100, 300, 500],
            "max_depth": [5, 10, None],
            "min_samples_leaf": [1, 2, 5],
            "max_features": ["sqrt", 0.5, 1.0]
        }
    },

    "Neural Network": {
        "estimator": TransformedTargetRegressor(
            regressor=Pipeline(steps=[
                ("scaler", StandardScaler()),
                ("model", MLPRegressor(
                    activation="relu",
                    solver="adam",
                    max_iter=3000,
                    early_stopping=True,
                    n_iter_no_change=20,
                    random_state=42,
                    learning_rate="adaptive"
                ))
            ]),
            transformer=StandardScaler()
        ),
        "param_grid": {
            "regressor__model__hidden_layer_sizes": [
                (16,),
                (32,),
                (32, 16),
                (64, 32),
                (64, 32, 16),
                (128, 64, 32, 16, 8)
            ],
            "regressor__model__learning_rate_init": [
                0.001,
                0.0005,
                0.0001
            ],
            "regressor__model__alpha": [
                0.0001,
                0.001,
                0.01
            ]
        }
    }
}


# path for results
RESULTS = BASE / "results"

# initialize containers for all results
full_predictions = []
full_importances = []

target_col = "Stabilized Day Ahead Price [EUR/MWh]"


def run_hour(i):

    print(f"Start hour: {i}")

    # get current hour data frame
    df_model = hourly_dfs[i].copy()

    # sort by timestamp
    df_model = df_model.sort_values("timestamp").reset_index(drop=True)

    # set features
    feature_cols = df_model.columns[
        ~df_model.columns.isin([target_col, "timestamp", "Day Ahead Price [EUR/MWh]"])
    ].tolist()

    # set collector lists
    all_predictions = []
    all_feature_importances = []

    # determine first test day
    first_test_day = df_model["timestamp"].min() + pd.DateOffset(years=2)

    # set last possible test day
    last_test_day = pd.to_datetime("04.06.2026  00:00:00")

    # initialize current test day
    current_test_day = first_test_day

    # set hpo dicts
    hpo_done = False
    best_params_by_model = {}
    best_cv_score_by_model = {}
    best_cv_mae_by_model = {}

    # perform rolling forecast
    while current_test_day <= last_test_day:

        print(f"Hour {i} | Test day: {current_test_day.date()}")

        # train set: prior year
        train_start = current_test_day - pd.DateOffset(years=2)
        train_end = current_test_day

        train = df_model[
            (df_model["timestamp"] >= train_start) &
            (df_model["timestamp"] < train_end)
        ].copy()

        # test set: next day
        test_start = current_test_day
        test_end = current_test_day + pd.DateOffset(days=1)

        test = df_model[
            (df_model["timestamp"] >= test_start) &
            (df_model["timestamp"] < test_end)
        ].copy()

        # skip if no data
        if len(train) == 0 or len(test) == 0:
            current_test_day += pd.DateOffset(days=1)
            continue

        if len(train) == 0 or len(test) == 0:
            current_test_day += pd.DateOffset(days=1)
            continue

        X_train = train[feature_cols]
        y_train = train[target_col]

        X_test = test[feature_cols]
        y_test = test[target_col]

        # hpo on first training intervall
        if not hpo_done:

            print(
                f"Hour {i} | Running HPO only once on first valid interval: "
                f"{current_test_day.date()}"
            )

            # time series cross-validation only on first training set
            tscv = TimeSeriesSplit(n_splits=N_SPLITS)

            for model_name, setup in GRID_SEARCH_SPACES.items():

                estimator = setup["estimator"]
                param_grid = setup["param_grid"]

                grid_search = GridSearchCV(
                    estimator=estimator,
                    param_grid=param_grid,
                    scoring=SCORING,
                    cv=tscv,
                    n_jobs=GRID_N_JOBS,
                    refit=True
                )

                grid_search.fit(X_train, y_train)

                best_params_by_model[model_name] = grid_search.best_params_
                best_cv_score_by_model[model_name] = grid_search.best_score_
                best_cv_mae_by_model[model_name] = -grid_search.best_score_

                print(
                    f"Hour {i} | {model_name} | "
                    f"Best CV MAE: {-grid_search.best_score_:.4f} | "
                    f"Best params: {grid_search.best_params_}"
                )

            hpo_done = True

        # train models with optimized hpos
        for model_name, setup in GRID_SEARCH_SPACES.items():

            estimator = setup["estimator"]

            # fresh estimator copy for each rolling window
            model = clone(estimator)

            # use fixed parameters from first HPO interval
            model.set_params(**best_params_by_model[model_name])

            # fit model on current rolling training window
            model.fit(X_train, y_train)

            # predict
            y_pred = model.predict(X_test)

            # save predictions
            pred_df = pd.DataFrame({
                "timestamp": test["timestamp"].values,
                "model": model_name,
                "hour": i,
                "y_true": y_test.values,
                "y_pred": y_pred,
                "train_start": train_start,
                "train_end": train_end,
                "test_day": test_start,
                "train_rows": len(train),
                "test_rows": len(test),
                "best_cv_score_neg_mae": best_cv_score_by_model[model_name],
                "best_cv_mae": best_cv_mae_by_model[model_name],
                "best_params": str(best_params_by_model[model_name])
            })

            all_predictions.append(pred_df)

            # Feature Importance
            if model_name in ["Random Forest", "Decision Tree"]:

                importance_values = model.feature_importances_

                fi_df = pd.DataFrame({
                    "test_day": test_start,
                    "model": model_name,
                    "hour": i,
                    "feature": feature_cols,
                    "importance": importance_values,
                    "importance_type": "tree_feature_importance",
                    "best_params": str(best_params_by_model[model_name]),
                    "best_cv_mae": best_cv_mae_by_model[model_name]
                })

                all_feature_importances.append(fi_df)
            elif model_name == "ARX":

                # coefficients from Lasso inside the pipeline
                coef_values = model.named_steps["model"].coef_

                fi_df = pd.DataFrame({
                    "test_day": test_start,
                    "model": model_name,
                    "hour": i,
                    "feature": feature_cols,
                    "importance": np.abs(coef_values),
                    "coefficient": coef_values,
                    "importance_type": "lasso_standardized_coefficient_abs",
                    "best_params": str(best_params_by_model[model_name]),
                    "best_cv_mae": best_cv_mae_by_model[model_name]
                })

                all_feature_importances.append(fi_df)

            elif model_name == "Neural Network":

                # permutation importance for neural network
                perm = permutation_importance(
                    model,
                    X_train,
                    y_train,
                    random_state=42,
                    scoring="neg_mean_absolute_error",
                    n_jobs=PERM_N_JOBS
                )

                fi_df = pd.DataFrame({
                    "test_day": test_start,
                    "model": model_name,
                    "hour": i,
                    "feature": feature_cols,
                    "importance": perm.importances_mean,
                    "importance_std": perm.importances_std,
                    "importance_type": "permutation_importance_train",
                    "best_params": str(best_params_by_model[model_name]),
                    "best_cv_mae": best_cv_mae_by_model[model_name]
                })

                all_feature_importances.append(fi_df)

        # Naive Baseline
        naive_prediction = y_train.iloc[-1]
        y_pred_naive = np.repeat(naive_prediction, len(y_test))

        pred_df_naive = pd.DataFrame({
            "timestamp": test["timestamp"].values,
            "model": "Naive",
            "hour": i,
            "y_true": y_test.values,
            "y_pred": y_pred_naive,
            "train_start": train_start,
            "train_end": train_end,
            "test_day": test_start,
            "train_rows": len(train),
            "test_rows": len(test),
            "best_cv_score_neg_mae": np.nan,
            "best_cv_mae": np.nan,
            "best_params": "not_applicable"
        })

        all_predictions.append(pred_df_naive)

        fi_naive = pd.DataFrame({
            "test_day": test_start,
            "model": "Naive",
            "hour": i,
            "feature": feature_cols,
            "importance": np.nan,
            "importance_type": "not_applicable",
            "best_params": "not_applicable",
            "best_cv_mae": np.nan
        })

        all_feature_importances.append(fi_naive)

        # go to next day
        current_test_day += pd.DateOffset(days=1)

    # get all predictions together for current hour
    rolling_predictions = pd.concat(all_predictions, ignore_index=True)

    # get all importances together for current hour
    feature_importances = pd.concat(all_feature_importances, ignore_index=True)

    # save results as csv
    feature_importances.to_csv(
        RESULTS / f"importances_{i}.csv",
        index=False
    )

    rolling_predictions.to_csv(
        RESULTS / f"predictions_{i}.csv",
        index=False
    )

    print(f"Finished hour: {i}")

    return {
        "hour": i,
        "predictions_path": str(RESULTS / f"predictions_{i}.csv"),
        "importances_path": str(RESULTS / f"importances_{i}.csv")
    }


# run all 24 hours in parallel
with threadpool_limits(limits=1):
    hour_results = Parallel(
        n_jobs=HOUR_N_JOBS,
        backend="loky",
        verbose=10
    )(
        delayed(run_hour)(i) for i in range(0, 24)
    )

hour_results
"""

'\n# settings\nSCORING = "neg_mean_absolute_error"\nHOUR_N_JOBS = 12\nGRID_N_JOBS = 2\nRF_N_JOBS = 4\nPERM_N_JOBS = 4\nN_SPLITS = 3\n\nGRID_SEARCH_SPACES = {\n\n    "Decision Tree": {\n        "estimator": DecisionTreeRegressor(\n            random_state=42\n        ),\n        "param_grid": {\n            "max_depth": [3, 5, 10, None],\n            "min_samples_leaf": [1, 5, 10, 20],\n            "min_samples_split": [2, 10, 20]\n        }\n    },\n\n    "ARX": {\n        "estimator": Pipeline(steps=[\n            ("scaler", StandardScaler()),\n            ("model", Lasso(\n                random_state=42,\n                max_iter=10000\n            ))\n        ]),\n        "param_grid": {\n            "model__alpha": [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]\n        }\n    },\n\n    "Random Forest": {\n        "estimator": RandomForestRegressor(\n            random_state=42,\n            n_jobs=RF_N_JOBS\n        ),\n        "param_grid": {\n            "n_estimators": [100, 30

### Load the results from the .csv files to save time.

In [22]:
BASE = Path(".")
RESULTS = BASE / "results"

# load predictions from csv results
pred_list = []
for i in range(24):
    temp_df = pd.read_csv(
        RESULTS / f"predictions_{i}.csv",
        parse_dates=["timestamp", "train_start", "train_end", "test_day"],
    )
    pred_list.append(temp_df)

full_predictions = pd.concat(pred_list, ignore_index=True)

# load importance from csv results
imp_list = []
for i in range(24):
    temp_df = pd.read_csv(RESULTS / f"importances_{i}.csv")
    imp_list.append(temp_df)

full_importances = pd.concat(imp_list, ignore_index=True)

In [23]:
def inverse_stabilization(y_pred_transformed, test_day, transform_params):
    """
    Inverse transformation of variance stabilization to compare tim-series in original data format:
        P = b * sinh(Y_hat) + a
    """
    a_i, b_i = transform_params[pd.Timestamp(test_day)]
    return b_i * np.sinh(y_pred_transformed) + a_i

In [24]:
## Application of inverse transformation

preds_real = full_predictions.copy()

preds_real["y_pred_real"] = np.nan
preds_real["y_true_real"] = np.nan

for idx, row in preds_real.iterrows():

    # get test day
    test_day = pd.Timestamp(row["test_day"]).floor("D")

    # get transformed values
    y_pred_t = row["y_pred"]
    y_true_t = row["y_true"]

    # get parameters of transformation
    if test_day not in transform_params:
        # if a day is missing -> skip
        continue

    a_i, b_i = transform_params[test_day]

    # inverse transformation
    preds_real.at[idx, "y_pred_real"] = b_i * np.sinh(y_pred_t) + a_i
    preds_real.at[idx, "y_true_real"] = b_i * np.sinh(y_true_t) + a_i

### Performance Evaluation

In [25]:
# generate model performance on original scale
model_performance_real = (
    preds_real.groupby(["model", "hour"])
    .apply(
        lambda x: pd.Series(
            {
                "MAE": mean_absolute_error(x["y_true_real"], x["y_pred_real"]),
                "RMSE": np.sqrt(mean_squared_error(x["y_true_real"], x["y_pred_real"])),
                "R2": r2_score(x["y_true_real"], x["y_pred_real"]),
                "Observations": len(x),
            }
        )
    )
    .reset_index()
    .sort_values("MAE")
)
model_performance_real

,model,hour,MAE,RMSE,R2,Observations
96,Random Forest,0,10.336234,16.562780,0.960638,2006.0
0,ARX,0,10.703101,16.788873,0.959556,2006.0
97,Random Forest,1,10.887047,17.304728,0.951358,2005.0
98,Random Forest,2,11.584127,18.482510,0.941872,2005.0
1,ARX,1,11.939721,18.533890,0.944203,2005.0
...,...,...,...,...,...,...
61,Naive,13,37.145287,58.062295,0.607603,2005.0
62,Naive,14,37.599307,60.084020,0.592147,2005.0
57,Naive,9,38.163200,59.233179,0.675808,2005.0
55,Naive,7,40.413665,62.597712,0.651796,2005.0


In [26]:
# combine all predictions into one DataFrame
full_predictions_df = preds_real.copy()
full_predictions_df = full_predictions_df.sort_values("timestamp").reset_index(
    drop=True
)

model_name = "Random Forest"

# filter for the selected model
temp = full_predictions_df[full_predictions_df["model"] == model_name]
temp = temp.sort_values("timestamp")

plt.figure(figsize=(18, 6))

# ACT prices
plt.plot(
    temp["timestamp"],
    temp["y_true_real"],
    label="Actual (real scale)",
    color="black",
    alpha=0.7,
)

# FC prices
plt.plot(
    temp["timestamp"],
    temp["y_pred_real"],
    label="Predicted (real scale)",
    color="royalblue",
    alpha=0.7,
)

# time-series plot
plt.title(f"Full Time Series – {model_name} (Real Scale)")
plt.xlabel("Timestamp")
plt.ylabel("Pice Day Ahead [€/MWh]")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("plots/time_series_comparison.png", dpi=100, bbox_inches="tight")
plt.close()

print("Comparison saved → plots/time_series_comparison.png")

Comparison saved → plots/time_series_comparison.png


In [27]:
# aggregate hourly model performance to overall model performance
model_performance_overall = (
    model_performance_real.dropna(subset=["MAE", "RMSE", "R2", "Observations"])
    .assign(
        weighted_MAE=lambda x: x["MAE"] * x["Observations"],
        weighted_RMSE_sq=lambda x: (x["RMSE"] ** 2) * x["Observations"],
    )
    .groupby("model", as_index=False)
    .agg(
        MAE_sum=("weighted_MAE", "sum"),
        RMSE_sq_sum=("weighted_RMSE_sq", "sum"),
        Observations=("Observations", "sum"),
    )
)

# calculate weighted overall metrics
model_performance_overall["MAE"] = (
    model_performance_overall["MAE_sum"] / model_performance_overall["Observations"]
)

model_performance_overall["RMSE"] = np.sqrt(
    model_performance_overall["RMSE_sq_sum"] / model_performance_overall["Observations"]
)
# keep relevant columns
model_performance_overall = (
    model_performance_overall[["model", "MAE", "RMSE", "Observations"]]
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_performance_overall

,model,MAE,RMSE,Observations
0,Random Forest,18.881482,31.551767,48121.0
1,Neural Network,22.333037,36.632139,48121.0
2,ARX,22.534373,38.389721,48121.0
3,Decision Tree,24.599315,40.856233,48121.0
4,Naive,31.810938,51.377580,48121.0


In [28]:
# pivot: rows = hours, columns = models
mae_by_hour_model = model_performance_real.pivot(
    index="hour", columns="model", values="MAE"
).sort_index()

# grouped bar chart
ax = mae_by_hour_model.plot(kind="bar", figsize=(18, 6), width=0.85)

plt.title("MAE per Hour and Model", fontsize=16, pad=12)
plt.xlabel("Hour of Day")
plt.ylabel("MAE [€/MWh]")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig("plots/mae_per_hour_grouped_by_model.png", dpi=100, bbox_inches="tight")
plt.close()

print("Grouped MAE plot saved → plots/mae_per_hour_grouped_by_model.png")

Grouped MAE plot saved → plots/mae_per_hour_grouped_by_model.png


In [29]:
# actual vs predicted time-series plot per model
for model_name in full_predictions_df["model"].unique():

    temp = full_predictions_df[full_predictions_df["model"] == model_name].copy()

    temp = temp.sort_values("timestamp")

    fig, ax = plt.subplots(figsize=(18, 6))

    ax.plot(
        temp["timestamp"],
        temp["y_true_real"],
        label="Actual",
        color="black",
        linewidth=1.5,
        alpha=0.75,
    )

    ax.plot(
        temp["timestamp"],
        temp["y_pred_real"],
        label="Predicted",
        color="royalblue",
        linewidth=1.5,
        alpha=0.75,
    )

    ax.set_title(
        f"Actual vs. Predicted Day-Ahead Price – {model_name}", fontsize=16, pad=12
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Day-Ahead Price [€/MWh]")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()

    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", model_name).strip("_")
    fname = f"plots/actual_vs_predicted_over_time_{safe_name}.png"

    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.close()

    print(f"{model_name} actual vs predicted time series saved → {fname}")

Decision Tree actual vs predicted time series saved → plots/actual_vs_predicted_over_time_Decision_Tree.png
ARX actual vs predicted time series saved → plots/actual_vs_predicted_over_time_ARX.png
Random Forest actual vs predicted time series saved → plots/actual_vs_predicted_over_time_Random_Forest.png
Neural Network actual vs predicted time series saved → plots/actual_vs_predicted_over_time_Neural_Network.png
Naive actual vs predicted time series saved → plots/actual_vs_predicted_over_time_Naive.png


In [30]:
# absolute error over time per model
full_predictions_df = preds_real.copy()

full_predictions_df["residual"] = (
    full_predictions_df["y_true_real"] - full_predictions_df["y_pred_real"]
)

full_predictions_df["abs_error"] = full_predictions_df["residual"].abs()

for model_name in full_predictions_df["model"].unique():

    temp = full_predictions_df[full_predictions_df["model"] == model_name].copy()

    temp = temp.sort_values("timestamp")

    fig, ax = plt.subplots(figsize=(18, 5))

    ax.plot(
        temp["timestamp"],
        temp["abs_error"],
        color="royalblue",
        linewidth=1.5,
        alpha=0.8,
    )

    mean_error = temp["abs_error"].mean()

    ax.axhline(
        mean_error,
        linestyle="--",
        color="black",
        linewidth=2,
        label=f"Mean absolute error: {mean_error:.2f}",
    )

    ax.set_title(
        f"Absolute Forecast Error over Time – {model_name}", fontsize=16, pad=12
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Absolute Error [€/MWh]")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()

    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", model_name).strip("_")
    fname = f"plots/absolute_error_over_time_{safe_name}.png"

    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.close()

    print(f"{model_name} absolute error plot saved → {fname}")

Decision Tree absolute error plot saved → plots/absolute_error_over_time_Decision_Tree.png
ARX absolute error plot saved → plots/absolute_error_over_time_ARX.png
Random Forest absolute error plot saved → plots/absolute_error_over_time_Random_Forest.png
Neural Network absolute error plot saved → plots/absolute_error_over_time_Neural_Network.png
Naive absolute error plot saved → plots/absolute_error_over_time_Naive.png


In [31]:
# generate average feature importance
avg_feature_importance = (
    full_importances.dropna(subset=["importance"])
    .groupby(["model", "feature"], as_index=False)
    .agg(mean_importance=("importance", "mean"), std_importance=("importance", "std"))
    .sort_values(["model", "mean_importance"], ascending=[True, False])
)

# create feature importance plot per model
for model_name in avg_feature_importance["model"].unique():

    temp = (
        avg_feature_importance[avg_feature_importance["model"] == model_name]
        .sort_values("mean_importance", ascending=True)
        .tail(15)
    )

    fig, ax = plt.subplots(figsize=(10, 7))

    ax.barh(temp["feature"], temp["mean_importance"], color="royalblue")

    ax.set_title(f"Top 15 Feature Importances – {model_name}", fontsize=16, pad=12)
    ax.set_xlabel("Mean Importance")
    ax.set_ylabel("Feature")
    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()

    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", model_name).strip("_")
    fname = f"plots/feature_importance_top15_{safe_name}.png"

    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.close()

    print(f"{model_name} feature importance plot saved → {fname}")

ARX feature importance plot saved → plots/feature_importance_top15_ARX.png
Decision Tree feature importance plot saved → plots/feature_importance_top15_Decision_Tree.png
Neural Network feature importance plot saved → plots/feature_importance_top15_Neural_Network.png
Random Forest feature importance plot saved → plots/feature_importance_top15_Random_Forest.png


### Interpretation

Looking at our supervised learning results for forecasting the Day-Ahead Electricity Price [in €/MWh] of the German-Luxembourg Bidding Zone, we can observe the following main components:

1. Model Setup and Benchmarking

   - In this section, we compare two baseline models and three machine learning models.
    - Baseline Models: 
    Naive Baseline: The naive model uses the last available observed price as prediction. It represents a simple persistence-based benchmark. 
    ARX Baseline: The ARX model serves as an additional statistical baseline. It combines autoregressive price information with exogenous input variables. This allows us to evaluate whether flexible machine learning models provide additional forecasting value beyond a classical autoregressive approach.
   - Machine Learning Models:
   Decision Tree
   Random Forest
   Neural Network
   - This setup allows a structured comparison between a simple benchmark, a statistical time-series baseline, and more flexible nonlinear machine learning models.

2. Overall Forecasting Performance

   - Looking at the overall model performance, the Random Forest achieves the best forecasting accuracy across all evaluated models.
   - The Random Forest reaches the lowest MAE with 18.88 €/MWh and the lowest RMSE with 31.55 €/MWh.
   - Compared to the naive baseline, which reaches an MAE of 31.81 €/MWh and an RMSE of 51.38 €/MWh, the Random Forest reduces the average absolute forecasting error by approximately 40.6%.
   - The Neural Network and the ARX baseline perform similarly, with MAE values of 22.33 €/MWh and 22.53 €/MWh.
   - Both models clearly outperform the naive baseline, which indicates that both market fundamentals and autoregressive structures provide relevant predictive information.
   - However, the Random Forest clearly outperforms the ARX baseline, which suggests that nonlinear relationships and interactions between the input variables are relevant for electricity price forecasting.
   - The Decision Tree performs worse than the Random Forest, which is expected because a single tree is more sensitive to individual splits and less robust than an ensemble of many trees.
   - In summary, the results show that all more advanced models improve over the naive benchmark, while the Random Forest provides the strongest overall forecasting performance.

3. Hourly Forecasting Performance

   - The hourly MAE comparison shows that forecasting accuracy differs substantially across the 24 delivery hours.
   - Across nearly all hours, the Random Forest provides the lowest prediction error.
   - This indicates that the strong overall performance of the Random Forest is not driven by only a few specific hours, but is relatively consistent across the full day.
   - Forecast errors are lowest during night and early morning hours. During these hours, electricity demand and price dynamics are usually more stable, which makes the day-ahead price easier to predict. The errors increase during daytime and evening hours. This pattern is visible across all models and suggests that prices are harder to forecast during periods with stronger load variation, changing renewable feed-in, and higher market activity.
   - The naive baseline performs particularly weak during these more volatile hours. This shows that simply carrying forward past prices is not sufficient when intraday market conditions change.
   - The Random Forest remains the best performing model in both low-error and high-error hours, which suggests that it is better able to capture nonlinear interactions between lagged prices, load forecasts, renewable generation forecasts, and calendar effects.

4. Actual vs. Predicted Price Development

   - Looking at the actual vs. predicted time-series plots, all models are able to capture the general price level and the broad development of the day-ahead price over time.
   - This is especially visible during the high-price period between 2021 and 2023, where the predicted prices broadly follow the increase in actual prices.
   - However, the models differ in how well they react to sharp price spikes and sudden negative price events. The Random Forest follows the actual price series most closely and shows a smoother but still responsive prediction path. The ARX baseline and the Neural Network also capture the overall price dynamics, but show stronger deviations during volatile market periods. The Decision Tree appears less stable, which is consistent with its weaker MAE and RMSE performance. The naive baseline visually follows the actual price series due to its definition. However, it reacts only with delay and therefore performs worse when prices change sharply.
    - In summary, the time-series plots show that supervised learning models are useful for forecasting normal price developments, but sudden extreme price events remain difficult to predict.

5. Forecast Errors over Time

   - The absolute forecast error plots show that forecasting errors are not evenly distributed over time.
   - For all models, most errors remain relatively close to their average level. However, a small number of extreme periods creates very large error spikes. These spikes are especially visible during volatile market phases and individual extreme price events.
   - The Random Forest has the lowest mean absolute error, but it still produces large errors during rare and extreme market situations.
   - This shows that the model improves average forecasting accuracy but cannot fully eliminate the challenge of predicting sudden market shocks.
   - Resulting the RMSE values are considerably higher than the MAE values for all models. This confirms that large individual errors have an important effect on model performance, because RMSE penalizes extreme deviations more strongly than MAE.
   - Therefore, the main forecasting challenge is not the prediction of regular price levels, but the prediction of rare and extreme deviations.
   - In beginning of 2022 there is an increase of error for every model. One plausible reason for that could be the beginn of the Ukrain War.
   - Scince end 2024 the prices are more volatile. 

6. Feature Importance and Economic Interpretation

   - The feature importance plots provide insight into which variables are most relevant for the prediction task.
   - For the Random Forest and the Decision Tree, lagged price features dominate the ranking. This indicates a strong autoregressive structure of the day-ahead electricity price. In other words, recent price levels contain important information about future price levels. In addition to lagged prices, market fundamentals such as Other Production, wind onshore production, wind offshore production, photovoltaic production, and load forecasts are relevant predictors. This supports the economic intuition from the motivation and causal inference sections. The day-ahead price is closely linked to the expected balance between demand and supply. When load is expected to be high or available supply is expected to be low, prices tend to increase.
   - The Neural Network shows a broader feature relevance structure, including renewable generation forecasts, load forecasts, Other Production, and calendar variables. This suggests that the Neural Network uses a more distributed combination of market fundamentals and temporal information.
   - For the ARX model, the most relevant feature is the previous-day price at the same delivery hour, which again confirms the strong autoregressive structure of day-ahead electricity prices. In addition, wind onshore production, Other Production, photovoltaic production and total load forecasts are among the most important predictors. This shows that the ARX model combines price persistence with exogenous market fundamentals to forecast hourly electricity prices.
   -  However, these feature importances should not be interpreted as causal effects, the feature importances describe predictive relevance rather than causal influence.

7. Economic Relevance

   - The improvement over the naive baseline is not only statistically relevant, but also economically meaningful. Day-ahead electricity prices influence short-term decisions of electricity producers, suppliers, traders, industrial consumers, and system operators. More accurate price forecasts can support bidding strategies, procurement decisions, risk management, and operational planning. The Random Forest reduces the MAE by approximately 40.6% compared to the naive benchmark. This suggests that the additional information from market fundamentals, renewable generation forecasts, load forecasts, calendar variables, and lagged prices creates relevant predictive value. From a practical perspective, lower forecast errors can reduce uncertainty in short-term electricity market decisions.
      - For producers, improved forecasts can support generation scheduling and bidding behavior.
      - For consumers and retailers, improved forecasts can help with procurement timing and price risk assessment.
   - However, the economic value should be interpreted carefully. The forecast improvements do not directly translate into profits, because actual financial gains would depend on transaction costs, bidding constraints, liquidity, market rules, risk limits, and the specific decision problem of the market participant. In addition, the largest errors occur during rare and extreme price events, which are often the most economically relevant situations. Therefore, the model provides clear average forecasting improvements, but its value in high-stress market conditions remains limited.

8. Limitations and Implications for following parts

   - Prediction-oriented interpretation
     - The supervised learning block is focused on forecasting performance and not on causal identification. The estimated relationships should therefore be interpreted as statistical patterns that are useful for prediction.
   - Unobserved influencing factors
     - Important drivers such as weather forecasts, fuel prices, CO₂ certificate prices, cross-border flows, power plant outages, and bidding behavior are not fully observed in the dataset. These omitted factors may explain why the models still struggle during extreme price events.
   - Strong autoregressive structure
     - Lagged price features are highly relevant for the prediction. This improves forecasting performance in regular market conditions. However, it can also limit the ability of the models to anticipate sudden structural breaks or rare shocks.
   - Model-specific feature importance
     - Feature importance values differ across model types. They may also be affected by correlations between features, especially among the many lagged price variables. Therefore, they should be interpreted as model-specific indicators of predictive relevance.
   - Hyperparameter optimization
     - Hyperparameter optimization is performed on the first rolling interval and then reused for the remaining rolling forecasts. This keeps the procedure computationally feasible. However, it may reduce the ability of the models to adapt to changing market conditions over time.
   - Implications for clustering analysis
     - The supervised learning results show that the day-ahead price can be predicted with meaningful accuracy, but that forecast errors increase during volatile and extreme periods. This motivates the following unsupervised learning section. By identifying recurring market regimes, the clustering analysis can help to better understand whether different price patterns are associated with different combinations of load, renewable generation, supply conditions, and calendar effects.


---
## Section 4 — Unsupervised / Generative Block
---
In this section, we aim to investigate the energy market in more detail to understand drivers of the day-ahead price. To achieve this, we identify market regimes for the Day-Ahead Price in the German-Luxembourg Bidding Zone by applying the k-means clustering technique. For this, the following pipeline is implemented:
1. Application of Elbow Method
    - Determine an appropriate value for parameter k by evaluating the inerita curve.
    - For this, the algorithm is run with 10 different initializations for each k.
2. Application of k-Means Clustering
    - Performing the actual clustering using the Euclidean distance measure.
    - Again, 10 different initializations are used to avoid poor local minima.
3. Evaluation and Visualization of Clustering
    - Using the silhouette score ton evaluate the clustering performance overall and in comparison to varying k values.
    - Visualizing the clustering using the t-SNE technique which preserves local distance structures of the high-dimensional space and keeps them in the two-dimensional representation of the clusters.
4. Analysis of Clusters
    - Identifying cluster patterns to determine domain-specific cluster explanations and interpretations using box plots, stacked bar charts, and histograms

### Elbow Method

In [ ]:
# reduce the columns which should be used for clustering
feature_cols = [
    "Wind Offshore Production FC [MWh]",
    "Wind Onshore Production FC [MWh]",
    "Photovoltaic Production FC [MWh]",
    "Other Production FC [MWh]",
    "Total Load FC [MWh]",
]

# build the corresponding data frame
df_clustering = df[feature_cols].copy()

# scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clustering)

# calculate the elbow method
inertias = []
cluster_range = range(1, 11)
for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# plot the elbow
plt.figure(figsize=(7, 4))
plt.plot(cluster_range, inertias, marker="o")
plt.xlabel("#Cluster")
plt.ylabel("Inertia")
plt.title("Elbow-Method for K-Means")
plt.grid(True)
plt.savefig("plots/elbow_method.png", dpi=100, bbox_inches="tight")
plt.close()
print("Elbow method plot saved \u2192 plots/elbow_method.png")

Elbow method plot saved → plots/elbow_method.png


Take-Away:

Looking at the Inerita curve, only a minimal elbow is visible, yet we identify k = 4 to be the most reasonable choice as it marks the point where additional clusters seem to only bring slight improvements.

### k-Means Clustering

In [33]:
# set cluster amount
n_clusters = 4

# Apply K-Means
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)

# add cluster column to data frame to perform cluster analysis
df["cluster"] = kmeans.fit_predict(X_scaled)

### Cluster Visualization and Evaluation

In [34]:
# Use t-sne to visualize clusters
tsne = TSNE(
    n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42
)

X_tsne = tsne.fit_transform(X_scaled)

# save results of t-sne in data frame
df["tsne_1"] = X_tsne[:, 0]
df["tsne_2"] = X_tsne[:, 1]

# visualize
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    df["tsne_1"], df["tsne_2"], c=df["cluster"], cmap="tab10", alpha=0.7, s=30
)

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("K-Means Cluster visualized by t-SNE")

cbar = plt.colorbar(scatter)
cbar.set_label("Cluster")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plots/tsne_clusters.png", dpi=100, bbox_inches="tight")
plt.close()
print("t-SNE plot saved \u2192 plots/tsne_clusters.png")

t-SNE plot saved → plots/tsne_clusters.png


In [35]:
## Silhouette Score calculation
sil_score = silhouette_score(X_scaled, df["cluster"])
print(f"Silhouette Score: {sil_score:.4f}")

Silhouette Score: 0.2997


### Cluster Analysis

In [ ]:
# Get box plots for all features
cols = [
    "Wind Offshore Production FC [MWh]",
    "Wind Onshore Production FC [MWh]",
    "Photovoltaic Production FC [MWh]",
    "Other Production FC [MWh]",
    "Total Load FC [MWh]",
    "Day Ahead Price [EUR/MWh]",
    "weekday",
    "hour",
    "month",
]
for feature in cols:
    data = [df.loc[df["cluster"] == cluster, feature].dropna() for cluster in range(4)]

    plt.figure(figsize=(8, 5))
    plt.boxplot(data, tick_labels=range(4))

    plt.xlabel("Cluster")
    plt.ylabel(feature)
    plt.title(f"Distribution of {feature} by Cluster")
    plt.grid(True)
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", feature).strip("_")

    plt.savefig(f"plots/clusters_boxplot_{safe_name}.png", dpi=100, bbox_inches="tight")
    plt.close()
    print(f"{safe_name} Boxplot saved \u2192 plots/clusters_boxplot_{safe_name}/.png")

Wind_Offshore_Production_FC_MWh Boxplot saved → plots/clusters_boxplot_Wind_Offshore_Production_FC_MWh/.png
Wind_Onshore_Production_FC_MWh Boxplot saved → plots/clusters_boxplot_Wind_Onshore_Production_FC_MWh/.png
Photovoltaik_Production_FC_MWh Boxplot saved → plots/clusters_boxplot_Photovoltaik_Production_FC_MWh/.png
Other_Production_FC_MWh Boxplot saved → plots/clusters_boxplot_Other_Production_FC_MWh/.png
Total_Load_FC_MWh Boxplot saved → plots/clusters_boxplot_Total_Load_FC_MWh/.png
Day_Ahead_Price_EUR_MWh Boxplot saved → plots/clusters_boxplot_Day_Ahead_Price_EUR_MWh/.png
weekday Boxplot saved → plots/clusters_boxplot_weekday/.png
hour Boxplot saved → plots/clusters_boxplot_hour/.png
month Boxplot saved → plots/clusters_boxplot_month/.png


In [37]:
# get histograms per feature and cluster
for feature in cols:
    plt.figure(figsize=(8, 5))

    for cluster in range(4):
        values = df.loc[df["cluster"] == cluster, feature].dropna()

        plt.hist(values, bins=30, alpha=0.5, label=f"Cluster {cluster}")

    plt.xlabel(feature)
    plt.ylabel("Quantity")
    plt.title(f"Histogram of {feature} by Cluster")
    plt.legend()
    plt.grid(True)
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", feature).strip("_")
    plt.savefig(f"plots/clusters_hist_{safe_name}.png", dpi=100, bbox_inches="tight")
    plt.close()
    print(f"{safe_name} Histogram saved \u2192 plots/clusters_hist_{safe_name}/.png")

Wind_Offshore_Production_FC_MWh Histogram saved → plots/clusters_hist_Wind_Offshore_Production_FC_MWh/.png
Wind_Onshore_Production_FC_MWh Histogram saved → plots/clusters_hist_Wind_Onshore_Production_FC_MWh/.png
Photovoltaik_Production_FC_MWh Histogram saved → plots/clusters_hist_Photovoltaik_Production_FC_MWh/.png
Other_Production_FC_MWh Histogram saved → plots/clusters_hist_Other_Production_FC_MWh/.png
Total_Load_FC_MWh Histogram saved → plots/clusters_hist_Total_Load_FC_MWh/.png
Day_Ahead_Price_EUR_MWh Histogram saved → plots/clusters_hist_Day_Ahead_Price_EUR_MWh/.png
weekday Histogram saved → plots/clusters_hist_weekday/.png
hour Histogram saved → plots/clusters_hist_hour/.png
month Histogram saved → plots/clusters_hist_month/.png


In [38]:
# settings
n_bins = 5
time_features = ["weekday", "hour", "month"]

for feature in cols:

    temp = df[[feature, "cluster"]].dropna().copy()

    # Zeitfeatures → NICHT binnnen, numerisch lassen
    if feature in time_features:
        temp["bin"] = temp[feature]

    # numerische Features → versuchen zu binnen
    elif pd.api.types.is_numeric_dtype(temp[feature]):
        try:
            temp["bin"] = pd.qcut(temp[feature], q=n_bins, duplicates="drop").astype(
                str
            )
        except:
            temp["bin"] = "all_values"

    # nicht-numerisch → direkt verwenden
    else:
        temp["bin"] = temp[feature].astype(str)

    # Cluster-Anteile
    plot_data = pd.crosstab(temp["cluster"], temp["bin"], normalize="index") * 100

    # Plot
    plot_data.plot(kind="bar", stacked=True, figsize=(10, 5))

    plt.xlabel("Cluster")
    plt.ylabel("Share in %")
    plt.title(f"Distribution of {feature} per Cluster")

    # Legende unten
    plt.legend(
        title=feature,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        frameon=False,
    )

    plt.grid(axis="y", alpha=0.3)
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", feature).strip("_")
    plt.savefig(
        f"plots/clusters_barchart_{safe_name}.png",
        dpi=120,
        bbox_inches="tight",
        pad_inches=0.3,
    )
    plt.close()
    print(f"{safe_name} Bar Chart saved → plots/clusters_barchart_{safe_name}.png")

Wind_Offshore_Production_FC_MWh Bar Chart saved → plots/clusters_barchart_Wind_Offshore_Production_FC_MWh.png
Wind_Onshore_Production_FC_MWh Bar Chart saved → plots/clusters_barchart_Wind_Onshore_Production_FC_MWh.png
Photovoltaik_Production_FC_MWh Bar Chart saved → plots/clusters_barchart_Photovoltaik_Production_FC_MWh.png
Other_Production_FC_MWh Bar Chart saved → plots/clusters_barchart_Other_Production_FC_MWh.png
Total_Load_FC_MWh Bar Chart saved → plots/clusters_barchart_Total_Load_FC_MWh.png
Day_Ahead_Price_EUR_MWh Bar Chart saved → plots/clusters_barchart_Day_Ahead_Price_EUR_MWh.png
weekday Bar Chart saved → plots/clusters_barchart_weekday.png
hour Bar Chart saved → plots/clusters_barchart_hour.png
month Bar Chart saved → plots/clusters_barchart_month.png


Take-Away
1. Clustering Approach
    - As we want to focus on market regimes regarding supply and demand at the electricity market, we restrict the variable set used for the clustering to those that describe the current market state rather than predictive or engineered features. Thus, we exclude the temporal-related lag and min and max features, which where explicitly built for the supervised learning part and its prediction, features for seasonality, and holiday-related features.
    - Based on this, the following features are used for the clustering:
        - Wind Offshore Production FC [MWh]
        - Wind Onshore Production FC [MWh]
        - Photovoltaic Production FC [MWh]
        - Other Production FC [MWh]
        - Total Load FC [MWh]
    - All numeric features are scaled prior to the clustering process, to ensure comparability across features, given that K‑Means relies on Euclidean distances.
2. Results of Elbow Method
    - The Elbow plot revealed that only a minimal elbow is visible for the Inerita curve. Yet we identify k = 4 to be the most suitable choice for k as additional clusters yield only slight improvements in the Inerita curve.
3. Analysis of Clustering Performance
    - Analyzing the cluster visually is done using the t-SNE technique which is able to preserve local distance structures of high-dimensional space and to project them into the two-dimensional representation space of the clusters. 
    - Looking at the resulting plot for our dataset, the four clusters have a clear visual separation which confirms that the chosen feature are able to capture four different electricity market regimes for the German-Luxembourg Bidding Zone. At the cluster borders, there are a few overlapping visual areas, which could indicate transitional periods between the different clusters.
    - An analysis of the silhouette score reveals similar insights, as a score of about 0.3 is reached. This score indicates that the clustering identified some meaningful structures but that there is still moderate overlap between the different clusters.
4. Economic Interpretation of resulting Clusters
    - Cluster 0: Low-Load Hours
        - Short Description of Cluster
            - This cluster represents hours with low electricity demand, which typically occur during nighttime and on weekends, as consumption patterns are low and renewable generation does not exhibit a dominant pattern.
        - Detailed Characterization of Cluster:
            - Low Total Load FC compared to all other clusters.
            - Weekend days occur more frequently.
            - Night Hours (22-6) are overrepresented.
            - Summer months appear more often than in other clusters
            - Day‑Ahead Price at an average level: higher than in renewable‑dominated clusters, lower than in high‑load periods.
            - No systematic patterns in the production forecasts of wind, photovoltaic, or other production
    - Cluster 1: Wind (On- and Offshore)-dominated hours
        - Short Description of Cluster
            - Hours characterized by high wind generation, where onshore or offshore wind dominate the supply side and have a strong influence on decreasing market prices, sometimes enforcing negative electricity prices.
        - Detailed Characterization of Cluster:
            - High Wind (On- and Offshore) FC compared to all other clusters.
            - No systematic day patterns.
            - No systematic hour patterns.
            - Summer months are underrepresented, while fall to spring are more dominant.
            - Day‑Ahead Price at a rather low level: lower than non-renewable‑dominated clusters, higher than in photovoltaic-dominated periods.
            - No systematic patterns in the production forecasts of photovoltaic or other production
    - Cluster 2: Photovoltaic-dominated Hours
        - Short Description of Cluster
            - Hours characterized by high photovoltaic generation, which dominates the supply side and has a strong influence on decreasing market prices, which regularly enforce negative electricity prices.
        - Detailed Characterization of Cluster:
            - High Photovoltaic FC compared to all other clusters.
            - No systematic day patterns.
            - Strong intraday structure: Hours between 19 and 7 are absent, while midday hours dominate the cluster.
            - Seasonal pattern: Summer months are strongly overrepresented, winter months almost absent; spring and fall appear less frequently.                
            - Day‑Ahead Price at the lowest level of all clusters.
            - No systematic patterns in the production forecasts of wind or other production.
    - Cluster 3: High-Load Hours
        - Short Description of Cluster
            - Hours characterized by high electricity demand, which typically occur during daytime on weekdays and in colder seasons, where conventional generation plays a central role and market prices reach their highest levels.
        - Detailed Characterization of Cluster:
            - High Total Load FC compared to all other clusters.
            - Highes Other Production FC compared to all other clusters.
            - Week days occur more frequently.
            - Day hours are overrepresented with midday hours being slightly less present.
            - Fall to spring months appear more often than in other clusters.
            - Day‑Ahead Price at the highest level of all clusters.
            - No systematic patterns in the production forecasts of wind or photovoltaic production.
5. Comparison of results for varying k
    - To assess the robustness of the clustering, K‑Means was evaluated for several values of k. 
    - Across all tested values, the overall cluster patterns remained rather consistent, but the interpretability and separation quality varied:
        - k = 3:
            - The algorithm merged the two non-renewable clusters into one joint cluster, which masked the difference between high and low load hours given by the k=4 clusters above, leading to a worse/broader economical interpretation.
            - The clustering quality however, only drops slightly from 0.3 to 0.296, indicating only a slightly worse clustering performance.
        - k = 5:
            - The algorithm splits up the wind cluster into two distinct clusters for high and low load forecasts, which does not really add much value to our economic interpretation.
            - The clustering also drops stronger from 0.3 to 0.285, indicating a worse clustering performance.
    -  Overall, k = 4 was selected as the most appropriate choice based on the results of the elbow method as well as the economical interpretations as it provides a fundamental representation of the underlying market regimes. 
6. Limitations of applied Clustering
    - Restricted feature set
        - Clustering relies only on load and generation forecasts. Other price‑relevant drivers, such as cross‑border flows, outages, or other energy prices are not included. This limits the completeness of the regime representation.
    - Model assumptions of K‑Means
        - K‑Means assumes convex, spherical clusters and equal variance across dimensions. However, the electricity market data may not suit these assumptions in high-dimensional space.
    - Sensitivity to scaling and initialization
        - Although we use scaling and a fixed random state is used, K‑Means may converge to local minima. This can affect the cluster boundaries and separation quality. 
    - Moderate silhouette scores
        - Silhouette values indicate partial overlap between the defined clusters, which reflects that the electricity market regimes transition to each other rather than forming perfectly distinct states.



---
## Section 5 — Synthesis & Communication

### What causal inference revealed

* The causal inference block shows that the relationship between load forecasts, supply forecasts, renewable generation forecasts and the day-ahead price cannot be interpreted as fully causal in this study. The DAG indicates that important variables such as weather forecasts, cross-border flows, fuel prices and power plant outages are not fully observed. Especially the weather forecast acts as an unobserved confounder, since it affects both demand and renewable generation.
As a result, relevant backdoor paths remain open and a valid causal effect cannot be identified with the available data. Therefore, the following supervised learning results are interpreted as associative and prediction-oriented rather than causal.

---

### What supervised learning revealed

* The supervised learning block shows that hourly day-ahead prices can be forecasted with meaningful accuracy. The Random Forest performs best across the evaluated models with an MAE of 18.88 €/MWh and an RMSE of 31.55 €/MWh. Compared to the naive baseline, this reduces the MAE by approximately 40.6%.
The ARX model serves as an additional statistical baseline and performs similarly to the Neural Network. However, both are outperformed by the Random Forest, which suggests that nonlinear relationships between lagged prices, load, renewable generation and calendar effects are relevant for forecasting.
The hourly results show that errors are lower during night and early morning hours and higher during daytime and evening hours. The actual vs. predicted plots further show that the models capture regular price movements well, but still struggle with rare extreme price events.

---

### What clustering revealed

* The clustering block identifies recurring market regimes based on load forecasts, renewable generation forecasts and other production forecasts. Using k-means with k = 4, the analysis separates the market into four interpretable supply-demand regimes: low-load, wind-dominated, photovoltaic-dominated and high-load situations.
The t-SNE visualization and silhouette score indicate that the clusters contain meaningful structure, although they are not perfectly separated. The sensitivity analysis supports k = 4, as k = 3 merges relevant market regimes and k = 5 mainly splits the wind cluster without adding substantial additional economic insight.


---

### Limitations & honest discussion

* The main limitation is that the study is prediction-oriented and not causal. Important drivers such as weather forecasts, cross-border flows, fuel prices, CO₂ prices, power plant outages and bidding behavior are not fully observed.
The models also struggle during rare and extreme price events. Although the Random Forest improves average forecasting accuracy, large errors remain in volatile market phases. In addition, feature importances should be interpreted carefully, as they show predictive relevance but not causal influence.
The clustering results also have limitations. K-means assumes distance-based cluster structures, and the silhouette score shows that some overlap between clusters remains. Therefore, the clusters should be interpreted as approximate market regimes.

---

### Conclusion

* Overall, the research question can be answered positively. Hourly day-ahead electricity prices in the German-Luxembourg bidding zone can be forecasted with meaningful accuracy using market fundamentals, renewable generation forecasts, load forecasts, calendar variables and lagged prices.
The Random Forest provides the strongest performance and clearly improves over both the naive baseline and the ARX baseline. This indicates that the included variables contain relevant predictive information. However, the results should be interpreted as associative rather than causal, and extreme price events remain difficult to predict.

## References

Ghelasi, P., & Ziel, F. (2024). Hierarchical forecasting for aggregated curves with an application to day-ahead electricity price auctions. *International Journal of Forecasting*, *40(2)*, 581–596. https://doi.org/10.1016/j.ijforecast.2022.11.004

Harris, C. R., Millman, K. J., Van Der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., Van Kerkwijk, M. H., Brett, M., Haldane, A., Del Río, J. F., Wiebe, M., Peterson, P., … Oliphant, T. E. (2020). Array programming with NumPy. *Nature*, *585(7825)*, 357–362. https://doi.org/10.1038/s41586-020-2649-2

Hunter, J. D. (2007). Matplotlib: A 2D Graphics Environment. *Computing in Science & Engineering*, *9(3)*, 90–95. https://doi.org/10.1109/MCSE.2007.55

Macedo, D. P., Marques, A. C., & Damette, O. (2022). The role of electricity flows and renewable electricity production in the behaviour of electricity prices in Spain. *Economic Analysis and Policy*, *76*, 885–900. https://doi.org/10.1016/j.eap.2022.10.001

Maciejowska, K., Uniejewski, B., & Weron, R. (2023). Forecasting Electricity Prices. https://doi.org/10.1093/acrefore/9780190625979.013.667

McKinney, W. (2010). Data Structures for Statistical Computing in Python. 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825-2830.

Trebbien, J., Tausendfreund, A., Rydin Gorjão, L., & Witthaut, D. (2024). Patterns and correlations in European electricity prices. *Chaos: An Interdisciplinary Journal of Nonlinear Science*, *34(7)*, 073108. https://doi.org/10.1063/5.0201734

Murza, S., Siripanich, P., & Yakovets, A. (2026). vacanza/holidays: v0.99 (v0.99). Zenodo. https://doi.org/10.5281/zenodo.20708292

Waskom, M. (2021). seaborn: Statistical data visualization. *Journal of Open Source Software*, *6(60)*, 3021. https://doi.org/10.21105/joss.03021

Ziel, F., & Weron, R. (2018). Day-ahead electricity price forecasting with high-dimensional structures: Univariate vs. multivariate modeling frameworks. *Energy Economics*, *70*, 396–420. https://doi.org/10.1016/j.eneco.2017.12.016


Dataset: SMARD Marktdaten. Retrieved from https://www.smard.de/home/downloadcenter/download-marktdaten. Accessed 13.06.2026.